# Regional AR(1) models of B

Uses the same B, winters, blocking definitions, 5-day test and figure style as `2_dim/2_dim_pipeline.ipynb`, and reads that notebook's saved results.

In [1]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path[:0] = ["../2_dim", "../src"]
from ar1_sector_model import lib_sb_sbe
from blocking_index import CBL_FIXED
from coarsen_theta_long import build_matrix
from e1_native_model import load
from e1_theta_model import band_weights

THETA_DIR = os.path.expanduser("~/projects/THETA_DIR")   # machine specific: your copy of the daily theta files
PIPELINE_OUT = "../2_dim/notebook_output"
OUT_DIR = "../results/regional_ar1"
os.makedirs(OUT_DIR, exist_ok=True)

# Same settings as the pipeline. A winter is labelled by the year of its December.
ALL_YEARS = list(range(1957, 2002))
TRAIN_YEARS, TEST_YEARS = ALL_YEARS[:30], ALL_YEARS[30:]
LAT_RANGE = (25.0, 76.0)
LONS = np.arange(0, 360, 5)
N_LON = LONS.size
COARSEN_FINE_DEG, COARSEN_HALF_WIDTH_DEG = 0.25, 2.5
HALF_BAND_DEG, BAND_FINE_DEG = 15.0, 0.25
NDAYS, STRIDE, NSIM, N_SD = 5, 1, 10, 2
PHI, TAPER_DEG = 0.62, 15
SECTOR_MASK = np.isin(LONS, np.r_[np.arange(330, 360, 5), np.arange(0, 95, 5)])
MEASURES = ["LIB", "SB", "SBE"]
nw = len(TEST_YEARS)

## B

B at one fixed central latitude per longitude, computed from daily theta with the pipeline's weights.

In [2]:
theta_winters, lats, lons_native = load(THETA_DIR, LAT_RANGE)
W_lon = build_matrix(nlon_in=lons_native.size, nlon_out=N_LON, fine=COARSEN_FINE_DEG, half=COARSEN_HALF_WIDTH_DEG)
W_lat = np.stack([band_weights(lats, c, c + HALF_BAND_DEG, fine=BAND_FINE_DEG)
                  - band_weights(lats, c - HALF_BAND_DEG, c, fine=BAND_FINE_DEG) for c in CBL_FIXED])
W_B = (W_lat[:, :, None] * W_lon[:, None, :]).reshape(N_LON, -1)
B_winters = {y: th.reshape(th.shape[0], -1).astype(np.float64) @ W_B.T for y, _, th in theta_winters}
assert sorted(B_winters) == ALL_YEARS

mu = np.concatenate([B_winters[y] for y in TRAIN_YEARS]).mean(0)        # mean B at each longitude, training winters
x0 = np.concatenate([B_winters[y][:-1] - mu for y in TRAIN_YEARS])      # today, minus mu
x1 = np.concatenate([B_winters[y][1:] - mu for y in TRAIN_YEARS])       # tomorrow, minus mu

## Model and 5-day test

`B[t+1] - mu = phi (B[t] - mu) + eps`, with noise covariance limited to ±15°, as in the pipeline's MHW-style model. Each run starts from the observed state on a test day and simulates 5 days, 10 times per start.

In [3]:
def fit_ar1(phi):
    """phi: one value, or one per longitude. Noise comes from the training residuals."""
    res = x1 - phi * x0
    C = np.cov(res.T)
    d = np.abs(LONS[:, None] - LONS[None, :]) % 360
    d = np.minimum(d, 360 - d)
    w, V = np.linalg.eigh(np.where(d <= TAPER_DEG, C, 0.0))
    return dict(phi=phi, L=V * np.sqrt(np.clip(w, 0, None)))

def simulate(fit, y, starts, rng):
    """NSIM runs of NDAYS days from each start day of winter y: (runs, NDAYS, longitudes)."""
    x = np.repeat(B_winters[y][starts] - mu, NSIM, axis=0)
    B = np.empty((x.shape[0], NDAYS, N_LON))
    for j in range(NDAYS):
        x = fit["phi"] * x + rng.standard_normal(x.shape) @ fit["L"].T
        B[:, j] = x + mu
    return B

def window_starts(nd):
    return np.arange(0, nd - NDAYS, STRIDE)

def data_frequencies(years):
    """Per-winter frequencies, (winters, longitudes). LIB, SB: every day. SBE: share of 5-day windows."""
    F = {k: [] for k in MEASURES}
    for y in years:
        B_all = B_winters[y]
        lib, sb, _ = lib_sb_sbe(B_all[None], periodic=True)
        F["LIB"].append(lib[0].mean(0))
        F["SB"].append(sb[0].mean(0))
        windows = np.stack([B_all[D + 1: D + 1 + NDAYS] for D in window_starts(len(B_all))])
        F["SBE"].append(lib_sb_sbe(windows, periodic=True)[2].mean(0))
    return {k: np.array(v) for k, v in F.items()}

def model_frequencies(fit, years, seed=0):
    rng = np.random.default_rng(seed)
    F = {k: [] for k in MEASURES}
    for y in years:
        lib, sb, sbe = lib_sb_sbe(simulate(fit, y, window_starts(len(B_winters[y])), rng), periodic=True)
        F["LIB"].append(lib.mean((0, 1)))
        F["SB"].append(sb.mean((0, 1)))
        F["SBE"].append(sbe.mean(0))
    return {k: np.array(v) for k, v in F.items()}

Check against the pipeline's saved results: the data curves should match exactly, and the MHW-style curves up to tiny differences.

In [4]:
saved = np.load(os.path.join(PIPELINE_OUT, "winter_freqs_test.npz"))
assert list(saved["years"]) == TEST_YEARS
FREQ = {"data": data_frequencies(TEST_YEARS),
        "eigenbasis": {k: saved[f"eigenbasis_{k}"] for k in MEASURES},
        "mhw": {k: saved[f"mhw_{k}"] for k in MEASURES}}

mhw_check = model_frequencies(fit_ar1(PHI), TEST_YEARS)
for k in MEASURES:
    print(f"{k}: largest difference from the pipeline: data {np.abs(FREQ['data'][k] - saved[f'data_{k}']).max():.2e}, "
          f"MHW-style {np.abs(mhw_check[k] - saved[f'mhw_{k}']).max():.2e}")
    assert np.array_equal(FREQ["data"][k], saved[f"data_{k}"])

LIB: largest difference from the pipeline: data 0.00e+00, MHW-style 4.71e-04
SB: largest difference from the pipeline: data 0.00e+00, MHW-style 2.35e-04
SBE: largest difference from the pipeline: data 0.00e+00, MHW-style 0.00e+00


## Scores and figures

Same table and figure style as the pipeline; intervals resample whole test winters, as in `modeling_baselines.ipynb`.

In [5]:
STYLE = {"data":         dict(color="#2a78d6", marker="o", ls="-",  label="Data (days D+1...D+5)"),
         "eigenbasis":   dict(color="#eb6834", marker="s", ls="--", label="Eigenbasis AR (1) simulation (days D+1...D+5)"),
         "mhw":          dict(color="#1baf7a", marker="^", ls="-.", label="MHW-style AR (1) simulation (days D+1...D+5)"),
         "th08_regions": dict(color="#55554f", marker="D", ls="--", label="Regional AR (1) simulation, one color per region (days D+1...D+5)")}
LONG = {"LIB": "local instantaneous blocking (LIB)", "SB": "sector blocking (SB)", "SBE": "sector blocking episode (SBE)"}
ndays = sum(len(B_winters[y]) for y in TEST_YEARS)
nwin = sum(window_starts(len(B_winters[y])).size for y in TEST_YEARS)
SUB = {"LIB": f"data: all {ndays} test days; models: mean of {NSIM} runs per start, {nwin} windows x {NDAYS} days",
       "SB":  f"data: all {ndays} test days; models: mean of {NSIM} runs per start, {nwin} windows x {NDAYS} days",
       "SBE": f"data: {nwin} sliding {NDAYS}-day windows; models: mean of {NSIM} runs per start"}

def score_table(names):
    rows = []
    for k in MEASURES:
        data_mean = FREQ["data"][k].mean(0)
        data_se = FREQ["data"][k].std(0, ddof=1) / np.sqrt(nw)
        for m in ["data"] + names:
            f = FREQ[m][k].mean(0)
            rows.append(dict(index=k, series=m, all_72_lons=f.mean(), sector_330_90=f[SECTOR_MASK].mean(),
                             mean_se=(FREQ[m][k].std(0, ddof=1) / np.sqrt(nw)).mean(),
                             rmse_vs_data=np.nan if m == "data" else np.sqrt(((f - data_mean) ** 2).mean()),
                             pct_lons_inside_data_band=np.nan if m == "data" else
                             100 * np.mean(np.abs(f - data_mean) <= N_SD * data_se)))
    return pd.DataFrame(rows)

N_BOOT = 10_000
picks = np.random.default_rng(0).integers(0, nw, size=(N_BOOT, nw))

def rmse_differences(pairs):
    """95% interval for the RMSE difference. An interval that excludes 0 means a clear difference."""
    rows = []
    for k in MEASURES:
        d_boot = FREQ["data"][k][picks].mean(1)
        boot = {m: np.sqrt(((FREQ[m][k][picks].mean(1) - d_boot) ** 2).mean(-1)) for m in {m for p in pairs for m in p}}
        for a, b in pairs:
            lo, hi = np.percentile(boot[a] - boot[b], [2.5, 97.5])
            rows.append(dict(measure=k, comparison=f"{a} minus {b}", low=lo, high=hi, clear_difference=(lo > 0) or (hi < 0)))
    return pd.DataFrame(rows)

def plot_regions(ax, k):
    """Regional model curve, one color per region, with dashed lines at the region boundaries."""
    f = FREQ["th08_regions"][k].mean(0)
    ax.plot([], [], ms=4, lw=2, **STYLE["th08_regions"])                 # legend entry
    for name, sel in TH08_REGIONS.items():
        idx = np.flatnonzero(sel)
        pieces = np.split(idx, np.flatnonzero(np.diff(idx) > 1) + 1)     # a region crossing 0°E has two pieces
        for piece in pieces:
            ax.plot(LONS[piece], f[piece], ms=4, lw=2, **{**STYLE["th08_regions"], "color": REGION_COLORS[name], "label": None})
        ax.axvline(LONS[sel & ~np.roll(sel, 1)][0] - 2.5, color="#9a9a94", ls="--", lw=1, zorder=1)
        ax.text(LONS[max(pieces, key=len)].mean(), 1.01, name.replace(" ", "\n"), transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=8, color="#444440")

def plot_frequencies(names, file_suffix):
    for k in MEASURES:
        fig, ax = plt.subplots(figsize=(9, 5.8), facecolor="#fcfcfb")
        ax.set_facecolor("#fcfcfb")
        f = FREQ["data"][k].mean(0)
        se = FREQ["data"][k].std(0, ddof=1) / np.sqrt(nw)
        ax.fill_between(LONS, np.maximum(f - N_SD * se, 0), f + N_SD * se, color=STYLE["data"]["color"], alpha=0.25, lw=0)
        for m in ["data"] + names:
            if m == "th08_regions":
                plot_regions(ax, k)
            else:
                ax.plot(LONS, FREQ[m][k].mean(0), ms=4, lw=2, **STYLE[m])
        ax.set_xticks(np.arange(0, 361, 30)); ax.set_xlim(-3, 358); ax.set_ylim(0, None)
        ax.set_xlabel("longitude (deg E)"); ax.set_ylabel("frequency")
        ax.set_title(f"{LONG[k]} frequency, DJF, {nw} test winters\n{SUB[k]}\nband: ±{N_SD} sd across winters / sqrt({nw})", fontsize=11, pad=30)
        ax.grid(True, color="#e6e5e0", lw=0.8)
        for s in ("top", "right"): ax.spines[s].set_visible(False)
        ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14)); fig.tight_layout()
        fig.savefig(os.path.join(OUT_DIR, f"{k}_test_data_{file_suffix}.png"), dpi=150)
        plt.show()

## Regional memory

One φ per region, the slope of tomorrow on today over that region's longitudes in the training winters. Means and noise are as in the MHW-style model. Regions are read off Tyrlis & Hoskins Fig. 3b.

In [6]:
def in_region(start, end):
    return ((LONS - start) % 360) <= ((end - start) % 360)

def regional_phi(regions):
    phi = np.full(N_LON, np.nan)
    for sel in regions.values():
        phi[sel] = (x0[:, sel] * x1[:, sel]).sum() / (x0[:, sel] ** 2).sum()
    assert np.isfinite(phi).all()
    return phi

TH08_REGIONS = {name: in_region(*edges) for name, edges in {   # °E, inclusive
    "Atlantic": (300, 350), "Europe": (355, 35), "Asia": (40, 115), "Western Pacific": (120, 155),
    "Central Pacific": (160, 195), "Eastern Pacific": (200, 235), "North America": (240, 295)}.items()}

REGION_COLORS = {"Europe": "#c2185b", "Asia": "#d98a00", "Western Pacific": "#8a5cd6", "Central Pacific": "#8d6e10",
                 "Eastern Pacific": "#0e7c86", "North America": "#b0413e", "Atlantic": "#3f3f46"}

phi_th08 = regional_phi(TH08_REGIONS)
print(pd.Series({name: phi_th08[sel][0] for name, sel in TH08_REGIONS.items()}).round(3).to_string())

Atlantic           0.656
Europe             0.664
Asia               0.666
Western Pacific    0.726
Central Pacific    0.698
Eastern Pacific    0.653
North America      0.661


## Results on the test winters

In [7]:
FREQ["th08_regions"] = model_frequencies(fit_ar1(phi_th08), TEST_YEARS)
train = data_frequencies(TRAIN_YEARS)
FREQ["trivial"] = {k: np.repeat(train[k].mean(0)[None], nw, axis=0) for k in MEASURES}   # training-winter frequency

REGIONAL = ["mhw", "eigenbasis", "th08_regions"]
frequencies_test = pd.DataFrame({"lon": LONS})
for m in FREQ:
    for k in MEASURES:
        frequencies_test[f"{k}_{m}"] = FREQ[m][k].mean(0)
        frequencies_test[f"{k}_{m}_se"] = FREQ[m][k].std(0, ddof=1) / np.sqrt(nw)
frequencies_test.to_csv(os.path.join(OUT_DIR, "frequencies_test_regional_ar1.csv"), index=False, float_format="%.5f")

scores = score_table(["trivial"] + REGIONAL)
scores.to_csv(os.path.join(OUT_DIR, "scores_test_regional_ar1.csv"), index=False, float_format="%.5f")
scores.round(4)

,index,series,all_72_lons,sector_330_90,mean_se,rmse_vs_data,pct_lons_inside_data_band
0,LIB,data,0.0807,0.1475,0.0116,NaN,NaN
1,LIB,trivial,0.0920,0.1624,0.0000,0.0208,73.6111
2,LIB,mhw,0.0892,0.1545,0.0031,0.0128,87.5000
3,LIB,eigenbasis,0.0903,0.1593,0.0039,0.0132,84.7222
4,LIB,th08_regions,0.0963,0.1622,0.0037,0.0186,63.8889
5,SB,data,0.1370,0.2494,0.0172,NaN,NaN
6,SB,trivial,0.1541,0.2702,0.0000,0.0321,72.2222
7,SB,mhw,0.1381,0.2514,0.0046,0.0145,100.0000
8,SB,eigenbasis,0.1524,0.2828,0.0063,0.0242,95.8333
9,SB,th08_regions,0.1502,0.2675,0.0055,0.0192,97.2222


In [8]:
differences = rmse_differences([("th08_regions", "mhw"), ("th08_regions", "eigenbasis"), ("th08_regions", "trivial")])
differences.to_csv(os.path.join(OUT_DIR, "rmse_differences_test_regional_ar1.csv"), index=False, float_format="%.5f")
differences.round(4)

,measure,comparison,low,high,clear_difference
0,LIB,th08_regions minus mhw,0.0021,0.0060,True
1,LIB,th08_regions minus eigenbasis,0.0031,0.0057,True
2,LIB,th08_regions minus trivial,-0.0082,0.0014,False
3,SB,th08_regions minus mhw,-0.0050,0.0088,False
4,SB,th08_regions minus eigenbasis,-0.0084,0.0013,False
5,SB,th08_regions minus trivial,-0.0210,-0.0050,True
6,SBE,th08_regions minus mhw,-0.0065,-0.0011,True
7,SBE,th08_regions minus eigenbasis,-0.0010,0.0057,False
8,SBE,th08_regions minus trivial,-0.0128,0.0082,False


In [9]:
plot_frequencies(["mhw", "th08_regions"], "regional_ar1")